# Notebook 05: EM visual evidence (picoc capture)

This notebook is part of the **visual demonstrator** of the
EM-software triangulation methodology. It walks through a
single capture from the picoc target, showing how the raw EM
traces look and how baseline traces compare to anomalous-execution
traces.

## Capture setup

- **Target software**: picoc, a small open-source C interpreter.
- **Hardware**: an ARMv7 single-board computer.
- **EM probe**: near-field magnetic, positioned over the CPU package.
- **Oscilloscope**: 10 GHz sample rate, ~50 us window per execution
  (500002 samples per trace).
- **Calibration phase**: 100 baseline traces using CPU-bound,
  Memory-bound, and IO-bound microbenchmarks. These define the
  subsystem-activation centroids used by the agreement-scoring
  step of the methodology.
- **Operation phase**: 3364 traces captured during a fuzzing
  campaign against picoc, exercising an integer overflow
  vulnerability that yields anomalous (non-zero) exit codes on a
  subset of inputs.

## Disclaimer

This notebook is an independent demonstrator of the EM-software
triangulation methodology. The numerical figures here
(`n_traces`, cluster counts, anomaly rates) are *not* intended
to match the paper's reported metrics, which use a different
selection of inputs. The visual demonstrator and the
reproducibility artefact (notebooks 01-04) are complementary
but independent.


## Load the feature CSV

The CSV at `../data/picoc_features.csv` is produced by
`scripts/extract_features_from_waveforms.py` from the raw
`.npz` traces. The traces themselves are not redistributed in
this repository.


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import IFrame, display

FEATURES_CSV = Path("..") / "data" / "picoc_features.csv"
FIGURES_DIR = Path("..") / "figures" / "em_evidence"

df = pd.read_csv(FEATURES_CSV)
df.head()


## Descriptive statistics


In [ ]:
print(f"Total rows: {len(df)}")
print()
print("Rows per campaign phase:")
print(df["campaign_phase"].value_counts().to_string())
print()
print("Anomalous vs normal:")
counts = df["is_anomalous"].value_counts()
for value, n in counts.items():
    label = "anomalous" if value else "normal"
    pct = 100.0 * n / len(df)
    print(f"  {label:<10} {n:>6}  ({pct:5.2f}%)")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].hist(df["duration"].dropna(), bins=60, color="#1f4e79", edgecolor="white")
axes[0].set_xlabel("Execution duration (s)")
axes[0].set_ylabel("Number of traces")
axes[0].set_title("Distribution of execution duration")
axes[0].grid(True, linestyle=":", alpha=0.5)

anomalous = df[df["is_anomalous"]]
if len(anomalous) > 0:
    exit_counts = anomalous["exit_code"].value_counts().sort_index()
    axes[1].bar(exit_counts.index.astype(str), exit_counts.values, color="#a23b3b")
    axes[1].set_xlabel("Exit code")
    axes[1].set_ylabel("Number of anomalous traces")
    axes[1].set_title("Exit-code distribution within anomalous traces")
    axes[1].grid(True, axis="y", linestyle=":", alpha=0.5)
else:
    axes[1].text(0.5, 0.5, "no anomalous traces", ha="center", va="center")

fig.tight_layout()
plt.show()


## Reference figure: baseline calibration profile

The figure below shows the baseline EM profile of the target
board in idle, with reference traces for CPU-bound,
Memory-bound, and IO-bound microbenchmarks. These centroids
define the subsystem-attribution criterion used in agreement
scoring.


In [ ]:
display(IFrame(str(FIGURES_DIR / "em_calibration_profile.pdf"), width=800, height=600))


## Reference figure: normal vs. anomalous mean traces

Mean EM trace comparison between normal-execution inputs
(`exit_code == 0`) and anomalous-execution inputs
(`exit_code != 0`). Visual divergence in the mid-band energy is
consistent with the Memory-subsystem activation predicted by
the methodology for this class of integer-overflow trigger.


In [ ]:
display(IFrame(str(FIGURES_DIR / "em_normal_vs_anomalous.pdf"), width=800, height=600))


## Next steps

Notebook 06 projects the 21-dimensional feature space into
two dimensions via PCA and t-SNE, and shows the corresponding
reference figures from the EM analysis pipeline.

Notebook 07 covers the diagnostic outputs of the EM-side
classifier and reproduces a baseline detection-metric report.
